# Module 3.2: Memory Identification

Not every piece of information deserves to be remembered. This notebook implements
explicit criteria for what qualifies as a **memory** vs transient information, and
builds an LLM-based classifier to make that determination automatically.

## What Makes Something Memorable?

| Criterion | Definition | Example |
|-----------|------------|----------|
| **Durability** | How likely to remain true | "I'm vegetarian" (high) vs "I'm in a rush today" (low) |
| **Reusability** | How likely to be useful later | "Prefers aisle seats" (high) vs "Flight UA123" (low) |
| **User-Specificity** | How personal vs generic | "Sarah hates layovers" (high) vs "JFK has 6 terminals" (low) |
| **Actionability** | Can agent use this to improve service | "Budget max $300/night" (high) vs "Weather is nice" (low) |

## Reference

> *"Towards Root Memories"* (arXiv:2606.23283) — introduces the distinction between
> reusable, decision-affecting "root" memories vs noise. Our identification criteria
> are inspired by their characterisation of memories that change downstream behaviour.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, json, asyncio
import nest_asyncio

sys.path.insert(0, "..")
nest_asyncio.apply()

from shared.travel_agent import create_client
from lifecycle_utils import MemoryCandidate, MemoryDecision

client, credential = create_client("../.env")
print("Client ready")

## The Memory Identification Prompt

We use the LLM itself to classify whether conversation turns contain memorable
information. The classifier evaluates each turn against our four criteria and
produces a structured `MemoryCandidate` output.

In [ ]:
IDENTIFICATION_PROMPT = """
You are a memory identification system. Given a conversation turn from the user,
determine whether it contains information worth memorising for future interactions.

Score each criterion 0.0 to 1.0:
- durability: Will this likely remain true for weeks/months?
- reusability: Will this be useful in future conversations?
- user_specificity: Is this personal to this user (not generic knowledge)?
- actionability: Can the agent use this to improve service?

Categories: preference | fact | event | procedure

Decision rules:
- If average score >= 0.6 → memorise
- If average score >= 0.4 and < 0.6 → ask_user
- Otherwise → discard

ANTI-PATTERNS (always discard):
- One-off questions ("what time is it")
- Hypotheticals ("what if I went to Paris")
- Other people's information ("my colleague likes sushi")
- Session-specific context ("as I said earlier")
- Sensitive data that shouldn't persist (credit card numbers, passwords)

Respond with ONLY valid JSON:
{
  "content": "extracted fact to store",
  "category": "preference|fact|event|procedure",
  "durability": 0.0,
  "reusability": 0.0,
  "user_specificity": 0.0,
  "actionability": 0.0,
  "decision": "memorise|discard|ask_user",
  "reasoning": "brief explanation"
}
"""
print("Identification prompt defined")

## Memory Identifier Class

Wraps the LLM classification into a reusable component that takes a user message
and returns a `MemoryCandidate` with structured scores.

In [ ]:
from agent_framework.foundry import FoundryChatClient

class MemoryIdentifier:
    """LLM-based classifier that determines if a user turn contains memorable info."""

    def __init__(self, client: FoundryChatClient, threshold: float = 0.6,
                 ask_threshold: float = 0.4):
        self.client = client
        self.threshold = threshold
        self.ask_threshold = ask_threshold

    async def evaluate(self, user_message: str) -> MemoryCandidate:
        """Evaluate a single user message for memorability."""
        response = await self.client.complete(
            messages=[
                {"role": "system", "content": IDENTIFICATION_PROMPT},
                {"role": "user", "content": user_message},
            ],
            temperature=0.1,  # Low temp for consistent classification
        )
        raw = response.choices[0].message.content.strip()
        # Parse JSON response
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(raw)
        return MemoryCandidate(
            content=data["content"],
            category=data["category"],
            durability=data["durability"],
            reusability=data["reusability"],
            user_specificity=data["user_specificity"],
            actionability=data["actionability"],
            decision=MemoryDecision(data["decision"]),
            reasoning=data["reasoning"],
        )

    async def evaluate_conversation(self, messages: list[str]) -> list[MemoryCandidate]:
        """Evaluate all user turns in a conversation."""
        candidates = []
        for msg in messages:
            candidate = await self.evaluate(msg)
            candidates.append(candidate)
        return candidates

identifier = MemoryIdentifier(client)
print("MemoryIdentifier ready")

## Demo: Classifying a Conversation

Let's take a realistic 10-turn travel conversation and see which turns produce
memory candidates. Watch how the classifier distinguishes durable preferences
from transient requests.

In [ ]:
# A realistic conversation between Sarah and the travel agent
SAMPLE_CONVERSATION = [
    # Turn 1: Durable preference (should memorise)
    "I always prefer window seats on long flights — I like watching the landscape.",
    # Turn 2: Transient request (should discard)
    "Can you check if there's a flight to Chicago tomorrow?",
    # Turn 3: Durable fact (should memorise)
    "I'm based in San Francisco, so SFO is my home airport.",
    # Turn 4: Hypothetical (should discard)
    "What if I wanted to fly first class — how much more would that be?",
    # Turn 5: Strong preference (should memorise)
    "I really don't like layovers longer than 2 hours. I'd rather pay more for direct.",
    # Turn 6: Other person's info (should discard)
    "My colleague Mike says the Hilton downtown is terrible.",
    # Turn 7: Actionable constraint (should memorise)
    "My company reimburses up to $250/night for hotels, so keep it under that.",
    # Turn 8: Session-specific (should discard)
    "Actually, go back to that first option you showed me.",
    # Turn 9: Repeated preference confirmation (should memorise)
    "Yes, Marriott is my go-to chain. I have their loyalty program.",
    # Turn 10: Sensitive data (should discard)
    "My loyalty number is MR-998877-2024.",
]

print(f"Evaluating {len(SAMPLE_CONVERSATION)} conversation turns...\n")

async def run_identification():
    candidates = await identifier.evaluate_conversation(SAMPLE_CONVERSATION)
    for i, (msg, candidate) in enumerate(zip(SAMPLE_CONVERSATION, candidates), 1):
        icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[
            candidate.decision.value
        ]
        print(f"Turn {i:2d} {icon} [{candidate.decision.value:8s}] "
              f"(score={candidate.composite_score:.2f})")
        print(f"         User: {msg[:60]}..." if len(msg) > 60 else f"         User: {msg}")
        if candidate.decision != MemoryDecision.DISCARD:
            print(f"         → Memory: {candidate.content}")
            print(f"           Category: {candidate.category} | {candidate.reasoning}")
        print()
    return candidates

candidates = asyncio.run(run_identification())

## Score Distribution

Let's visualise the composite scores to see the clear separation between
memorable and transient information.

In [ ]:
# Print a simple text-based score chart
print("Composite Score by Turn:")
print("=" * 60)
for i, c in enumerate(candidates, 1):
    bar = "█" * int(c.composite_score * 40)
    decision_icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[
        c.decision.value
    ]
    print(f"Turn {i:2d} {decision_icon} |{bar:<40}| {c.composite_score:.2f}")

print("\n" + "=" * 60)
print(f"Threshold: memorise >= 0.6, ask_user >= 0.4")
print(f"\nMemories identified: {sum(1 for c in candidates if c.decision == MemoryDecision.MEMORISE)}")
print(f"Discarded: {sum(1 for c in candidates if c.decision == MemoryDecision.DISCARD)}")
print(f"Ask user: {sum(1 for c in candidates if c.decision == MemoryDecision.ASK_USER)}")

## Filtering Pipeline

In production, identified candidates pass through additional filters before storage:

1. **Category allowlist** — Only store certain categories (e.g., no "event" for privacy)
2. **Minimum confidence** — Below threshold, discard even if identified
3. **Deduplication** — Check if we already know this fact
4. **Rate limiting** — Don't store more than N memories per conversation

In [ ]:
from dataclasses import dataclass

@dataclass
class FilterConfig:
    allowed_categories: list = None
    min_composite_score: float = 0.5
    max_memories_per_conversation: int = 10

    def __post_init__(self):
        if self.allowed_categories is None:
            self.allowed_categories = ["preference", "fact", "procedure"]


class MemoryFilterPipeline:
    """Post-identification filtering before memory storage."""

    def __init__(self, config: FilterConfig = None):
        self.config = config or FilterConfig()

    def apply(self, candidates: list[MemoryCandidate]) -> list[MemoryCandidate]:
        """Filter candidates through the pipeline."""
        # Step 1: Only keep 'memorise' decisions
        kept = [c for c in candidates if c.decision == MemoryDecision.MEMORISE]

        # Step 2: Category allowlist
        kept = [c for c in kept if c.category in self.config.allowed_categories]

        # Step 3: Minimum score
        kept = [c for c in kept if c.composite_score >= self.config.min_composite_score]

        # Step 4: Rate limit (keep highest-scored)
        kept.sort(key=lambda c: c.composite_score, reverse=True)
        kept = kept[:self.config.max_memories_per_conversation]

        return kept


pipeline = MemoryFilterPipeline()
filtered = pipeline.apply(candidates)

print(f"Before filtering: {len(candidates)} candidates")
print(f"After filtering:  {len(filtered)} memories to store\n")
for c in filtered:
    print(f"  [{c.category:10s}] {c.content}")
    print(f"              Score: {c.composite_score:.2f} | {c.reasoning}\n")

## Edge Cases: What NOT to Memorise

Let's test the classifier against tricky edge cases that look like preferences
but should NOT be stored.

In [ ]:
EDGE_CASES = [
    # Looks like preference but is hypothetical
    "If I were to move to London, I'd probably want to fly British Airways.",
    # Looks like a fact but is about someone else
    "My boss always flies Delta — maybe I should try them too.",
    # Contains sensitive data that shouldn't persist
    "My passport number is AB1234567, expiring March 2028.",
    # Temporary state, not durable
    "I'm feeling sick today so I might cancel my trip.",
    # Genuine durable preference (control case — should memorise)
    "I have a severe peanut allergy — please always flag this for meal selection.",
]

async def test_edge_cases():
    print("Edge Case Classification:\n")
    for msg in EDGE_CASES:
        candidate = await identifier.evaluate(msg)
        icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[
            candidate.decision.value
        ]
        print(f"{icon} [{candidate.decision.value:8s}] \"{msg[:65]}...\"" 
              if len(msg) > 65 else f"{icon} [{candidate.decision.value:8s}] \"{msg}\"")
        print(f"   Score: {candidate.composite_score:.2f} | {candidate.reasoning}\n")

asyncio.run(test_edge_cases())

## Connecting to the Lifecycle

Identified memories don't go directly to "trusted" — they enter the lifecycle as
**candidates**. This is the handoff point to Notebook 03 (Staged Promotion):

```
User message → MemoryIdentifier → MemoryCandidate → FilterPipeline → MemoryItem(state=CANDIDATE)
                                                                          ↓
                                                              Notebook 03: Staged Promotion
```

In [ ]:
from lifecycle_utils import MemoryItem, MemoryState

def candidate_to_memory(candidate: MemoryCandidate, user_id: str) -> MemoryItem:
    """Convert an identified candidate into a lifecycle-managed MemoryItem."""
    return MemoryItem(
        user_id=user_id,
        content=candidate.content,
        category=candidate.category,
        state=MemoryState.CANDIDATE,  # Always starts as candidate!
        confidence=candidate.composite_score,
        source_type="llm_inference",
    )

# Convert our filtered candidates to MemoryItems
memory_items = [candidate_to_memory(c, "E001") for c in filtered]

print(f"Created {len(memory_items)} MemoryItems (all in CANDIDATE state):\n")
for m in memory_items:
    print(f"  [{m.state.value:10s}] {m.content}")
    print(f"              confidence={m.confidence:.2f} | category={m.category}")
    print(f"              → Needs confirmation before agent uses this\n")

## Key Takeaways

1. **Not everything is memory** — apply explicit identification criteria before storing
2. **Four dimensions** — durability, reusability, user-specificity, actionability
3. **Anti-patterns matter** — hypotheticals, others' info, sensitive data must be excluded
4. **Pipeline filtering** — category allowlist, score threshold, rate limiting, dedup
5. **Memory starts as candidate** — identification doesn't equal trust (→ Notebook 03)

## Next: Staged Promotion (Notebook 03)

Now that we can identify *what* to store, the next question is:
**when should the agent trust it?** Newly identified memories must earn confidence
through repeated confirmation before influencing agent behaviour.